# Covariate Data Preprocessing

This notebook builds the final **covariate file** for QTL analysis by combining known covariates, genotype principal components (PCs), and hidden factors.


## Description

Covariates correct for confounders so that detected QTLs reflect true genotype-phenotype associations rather than technical or population artifacts. This notebook assembles three kinds of covariates into one file:

1. **Known covariates** — measured variables from the fixed covariate file (e.g. `msex`, `age_death`, `pmi`, `study`).
2. **Genotype PCs** — the top PCs from the genotype PCA (`3_genotype_pca.ipynb`), which capture population structure.
3. **Hidden factors** — unobserved sources of variation (batch effects, technical noise) inferred from the phenotype residuals.

The two steps are: (1) **merge** the genotype PCs with the known covariates, and (2) **compute residuals** on the merged covariates and run **hidden-factor analysis** (PCA on the residuals, with the number of factors chosen by the Marchenko-Pastur distribution). The result is a single covariate file ready for TensorQTL.


## Input Files

| File | Description |
| --- | --- |
| `input/covariate/protocol_example.covariates.tsv` | Fixed/known covariates, one row per sample: `#id`, `msex`, `age_death`, `pmi`, `study`. |
| `output/genotype/genotype_pca/protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.unrelated.plink_qc.prune.pca.rds` | Genotype PCA model from `3_genotype_pca.ipynb`. |
| `output/genotype/genotype_pca/...prune.pca.scree.txt` | Scree table from `3_genotype_pca.ipynb`; used to pick how many genotype PCs to keep. |
| `output/rnaseq/protocol_example.rnaseq.bed.bed.gz` | Phenotype bed (from `1_phenotype_preprocessing.ipynb`); used to compute residuals for hidden-factor analysis. |


## Steps


### **Step 1.** Merge genotype PCs with known covariates

Merge the genotype PCA results with the known covariate file. `--k` sets how many genotype PCs to keep; here it is read from the scree table as the number of PCs that together explain < 80% of variance (15 for this dataset). `--tol-cov 0.4` is the maximum allowed missingness rate for covariates. The output is a merged covariate table (known covariates + genotype PCs).


In [ ]:
sos run pipeline/covariate_formatting.ipynb merge_genotype_pc \
    --cwd output/covariate \
    --pcaFile output/genotype/genotype_pca/protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.unrelated.plink_qc.prune.pca.rds \
    --covFile input/covariate/protocol_example.covariates.tsv \
    --tol-cov 0.4 \
    --k `awk '$3 < 0.8' output/genotype/genotype_pca/protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.unrelated.plink_qc.prune.pca.scree.txt | tail -1 | cut -f 1`


In [ ]:
cd /home/ubuntu/xqtl_protocol_exercise
head data/covariate/covariates.tsv

### **Step 2.** Compute residuals and run hidden-factor analysis

Using the phenotype and the merged covariates from Step 1, this step first regresses out the known covariates to obtain phenotype **residuals** (`Marchenko_PC_1`), then runs PCA on those residuals and keeps the components that pass the **Marchenko-Pastur** significance threshold as **hidden factors** (`Marchenko_PC_2`). `--mean-impute-missing` fills any missing covariate values with their column mean. The final output combines the known covariates, genotype PCs, and hidden factors — this is the covariate file used in downstream QTL analysis.


In [ ]:
sos run pipeline/covariate_hidden_factor.ipynb Marchenko_PC \
    --cwd output/covariate \
    --phenoFile output/rnaseq/protocol_example.rnaseq.bed.bed.gz \
    --covFile output/covariate/protocol_example.covariates.protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.unrelated.plink_qc.prune.pca.gz \
    --mean-impute-missing


In [ ]:
cd /home/ubuntu/xqtl_protocol_exercise
# assuming no related data in previous geno qc step using plink_qc.prune.pca.rds
sos run pipeline/covariate_formatting.ipynb merge_genotype_pc \
    --cwd output/covariate/ \
    --pcaFile output/genotype/genotype_pca/wgs.merged.plink_qc.plink_qc.prune.pca.rds \
    --covFile data/covariate/covariates.tsv \
    --tol-cov 0.4 \
    --k `awk '$3 < 0.8' output/genotype/genotype_pca/wgs.merged.plink_qc.plink_qc.prune.pca.scree.txt | tail -1 | cut -f 1 ` 
    

## Output Files

| File | Description |
| --- | --- |
| `output/covariate/protocol_example.covariates.protocol_example.genotype...prune.pca.gz` | Merged covariates: known covariates + genotype PCs (output of Step 1). |
| `output/covariate/protocol_example.rnaseq.bed.protocol_example.covariates...prune.pca.residual.bed.gz` | Phenotype residuals after regressing out known covariates (intermediate, Step 2). |
| `output/covariate/protocol_example.rnaseq.bed.protocol_example.covariates...prune.pca.Marchenko_PC.gz` | **Final covariate file**: known covariates + genotype PCs + hidden factors, ready for QTL analysis. |


## Anticipated Results

The final `*.Marchenko_PC.gz` file contains, as rows, the 4 known covariates (`msex`, `age_death`, `pmi`, `study`), 15 genotype PCs (`PC1`-`PC15`), and the hidden factors (`Hidden_Factor_PC1` onward) inferred by Marchenko-Pastur, with one column per sample. This single file is the covariate input for the downstream TensorQTL association analysis.


In [3]:
# preview of the final Marchenko_PC.gz file
zcat output/covariate/bulk_rnaseq_tmp_matrix.low_expression_filtered.outlier_removed.tmm.expression.covariates.wgs.merged.plink_qc.plink_qc.prune.pca.Marchenko_PC.gz | head | cut -f 1-6

#id	sample0	sample1	sample2	sample3	sample4
sex	0	0	1	0	1
age	91	92	52	85	54
rin	4	6	3	5	7
pmi	4	33	21	28	36
PC1	0.0079719403735707	0.01551892391210383	0.00774270222373719	0.01728777110680747	-0.0038685278561914
PC2	0.21108638503242705	-0.08141101340591093	-0.1324609481859247	0.01462193776343482	0.06596595896560435
PC3	-0.05063343099042864	-0.12687379700590845	-0.12060270181509317	0.05599049431828046	-0.06686964706351353
PC4	-0.01406533451346132	0.12781517725968655	-0.01736711354272593	0.02015639819991523	-0.03304252616108331
PC5	-0.124368907986055	0.03655636585203538	0.0282157547046425	0.1561247811903808	-0.07371828928862333


In [4]:
zcat output/covariate/bulk_rnaseq_tmp_matrix.low_expression_filtered.outlier_removed.tmm.expression.covariates.wgs.merged.plink_qc.plink_qc.prune.pca.Marchenko_PC.gz | cut -f 1


#id
sex
age
rin
pmi
PC1
PC2
PC3
PC4
PC5
PC6
PC7
PC8
PC9
PC10
PC11
PC12
PC13
PC14
PC15
Hidden_Factor_PC1
Hidden_Factor_PC2
Hidden_Factor_PC3
Hidden_Factor_PC4
Hidden_Factor_PC5
Hidden_Factor_PC6
Hidden_Factor_PC7
Hidden_Factor_PC8
Hidden_Factor_PC9
Hidden_Factor_PC10
